In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
#Load the Breast Cancer Wisconsin dataset using Scikit-learn.
from sklearn.datasets import load_breast_cancer
cancer = load_breast_cancer()
data = cancer
y = cancer.target

In [ ]:
cancer.keys()

In [ ]:
#convert dataset to dataframe
df = pd.DataFrame(cancer.data, columns=cancer.feature_names)
df_cancer = df.copy()

In [ ]:
df_cancer.head()

In [ ]:
df_cancer.tail()

In [ ]:
df_cancer.info()

In [ ]:
df_cancer.describe()

In [ ]:
df_cancer.isnull().sum()

In [ ]:
print(df_cancer.duplicated().sum())

In [ ]:
#define target variables
df['target'] = [data.target_names[i] for i in data.target]

In [ ]:
#distribution count
df_cancer.hist(figsize=(20,20))
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
ax = sns.countplot(data=df, x='target', hue='target', palette='Set2', legend=False)
plt.title('Class Distribution: Breast Cancer Wisconsin Dataset')
plt.xlabel('Class')
plt.ylabel('Count')
plt.show()

In [ ]:
#Perform data preprocessing
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_cancer)
display(X_scaled)


In [ ]:
#Apply PCA to reduce the dimensionality of the dataset
from sklearn.decomposition import PCA
pca = PCA(n_components=4)
X_pca = pca.fit_transform(X_scaled)
display(X_pca)


In [ ]:
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)
print(cumulative_variance)

In [ ]:
pca_full = PCA().fit(X_scaled)
full_cum = np.cumsum(pca_full.explained_variance_ratio_)
n90=np.argmax(full_cum>=0.90)+1
n95=np.argmax(full_cum>=0.95)+1
print(f"Number of components needed for 90% variance: {n90}")
print(f"Number of components needed for 95% variance: {n95}")

In [ ]:
component_names = ['PC 1', 'PC 2', 'PC 3', 'PC 4']
pca_df = pd.DataFrame(data=X_pca, columns=component_names)
pca_df['Target'] = y
pca_df['Diagnosis'] = [data.target_names[i] for i in y]
print(pca_df.head().to_string(index=False))

In [ ]:
exp_var_ratio = pca.explained_variance_ratio_ * 100
cum_var_ratio = np.cumsum(exp_var_ratio)
plt.figure(figsize=(10, 6))
bars = plt.bar(component_names, exp_var_ratio, color='#377eb8', alpha=0.7, label='Individual Variance')
plt.plot(component_names, cum_var_ratio, color='#ff7f00', linestyle='--', marker='o', label='Cumulative Variance')
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 1, f'{yval:.1f}%', ha='center', va='bottom', fontsize=10)
plt.title('Variance Explained by 4 Principal Components', fontsize=14, pad=15, fontweight='bold')
plt.xlabel('Principal Components', fontsize=12)
plt.ylabel('Percentage of Variance (%)', fontsize=12)
plt.ylim(0, 110)
plt.legend(loc='center right')
plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
#Apply LDA to project the data into a lower-dimensional space while considering class labels.
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
target_names = data.target_names
lda = LinearDiscriminantAnalysis(n_components=1)
X_lda = lda.fit_transform(X_scaled, y)
lda_df = pd.DataFrame(data=X_lda, columns=['LDA Component 1'])
lda_df['Target'] = y
lda_df['Diagnosis'] = [target_names[i] for i in y]
print(lda_df.head().to_string(index=False))

In [ ]:
sns.histplot(data=lda_df[lda_df['Target'] == 0], x='LDA Component 1',
             color='#e41a1c', label='Malignant', kde=True, alpha=0.6, bins=20)
sns.histplot(data=lda_df[lda_df['Target'] == 1], x='LDA Component 1',
             color='#377eb8', label='Benign', kde=True, alpha=0.6, bins=20)
plt.title('LDA Projection: Optimal 1D Class Separation', fontsize=14, pad=15, fontweight='bold')
plt.xlabel('Linear Discriminant 1', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.legend(title="Diagnosis")
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
#Apply ICA to extract statistically independent components from the dataset.
from sklearn.decomposition import FastICA
ica = FastICA(n_components=4, random_state=42, max_iter=1000)
X_ica = ica.fit_transform(X_scaled)
component_names = ['IC 1', 'IC 2', 'IC 3', 'IC 4']
ica_df = pd.DataFrame(data=X_ica, columns=component_names)
ica_df['Target'] = y
ica_df['Diagnosis'] = [target_names[i] for i in y]
print(ica_df.head().to_string(index=False))

In [ ]:
sns.set_theme(style="ticks")
g = sns.pairplot(ica_df, vars=component_names, hue="Diagnosis",
                 palette={'malignant': '#e41a1c', 'benign': '#377eb8'},
                 plot_kws={'alpha': 0.6, 's': 30, 'edgecolor': 'none'},
                 diag_kind='kde',
                 diag_kws={'alpha': 0.5})

g.fig.suptitle('ICA Projections: Pairwise Relationships of 4 Independent Components',
               y=1.02, fontsize=14, fontweight='bold')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
sns.scatterplot(data=pca_df, x='PC 1', y='PC 2', hue='Diagnosis', palette={'malignant': '#e41a1c', 'benign': '#377eb8'}, alpha=0.7, ax=axes[0])
axes[0].set_title('PCA: PC 1 vs PC 2', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Principal Component 1')
axes[0].set_ylabel('Principal Component 2')
axes[0].grid(True, linestyle='--', alpha=0.5)

In [ ]:
np.random.seed(42)
jitter = np.random.normal(0, 0.1, size=len(lda_df))
sns.scatterplot(x=lda_df['LDA Component 1'], y=jitter, hue=lda_df['Diagnosis'], palette={'malignant': '#e41a1c', 'benign': '#377eb8'}, alpha=0.7, ax=axes[1])
axes[1].set_title('LDA: Component 1 with Jitter', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Linear Discriminant 1')
axes[1].set_ylabel('Jitter')
axes[1].set_ylim(-0.5, 0.5)
axes[1].grid(True, linestyle='--', alpha=0.5)

In [ ]:
sns.scatterplot(data=ica_df, x='IC 1', y='IC 2', hue='Diagnosis', palette={'malignant': '#e41a1c', 'benign': '#377eb8'}, alpha=0.7, ax=axes[2])
axes[2].set_title('ICA: IC 1 vs IC 2', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Independent Component 1')
axes[2].set_ylabel('Independent Component 2')
axes[2].grid(True, linestyle='--', alpha=0.5)

plt.suptitle('Comparison of Dimensionality Reduction Transformations', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()